<a href="https://colab.research.google.com/github/AmeyaSingh23/AI-LAB/blob/main/1BF24CS040_Week7_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
def is_var(x):
    return isinstance(x, str) and x.startswith('?')

def match(pattern, fact, theta):
    """Matches a pattern (with variables) against a ground fact."""
    if theta is False:
        return False
    if len(pattern) != len(fact) or pattern[0] != fact[0]:
        return False

    new_theta = theta.copy()
    for p, f in zip(pattern[1:], fact[1:]):
        if is_var(p):
            if p in new_theta and new_theta[p] != f:
                return False
            new_theta[p] = f
        elif p != f:
            return False
    return new_theta

def match_all_premises(premises, facts, theta):
    """Recursively finds all valid substitutions (theta) for a list of premises."""
    if theta is False:
        return []
    if not premises:
        return [theta]

    first, rest = premises[0], premises[1:]
    results = []
    for fact in facts:
        new_theta = match(first, fact, theta)
        if new_theta is not False:
            results.extend(match_all_premises(rest, facts, new_theta))
    return results

def substitute(pattern, theta):
    """Applies variable substitutions to a pattern."""
    return tuple(theta.get(x, x) for x in pattern)

# --- Forward Chaining Engine ---
def forward_chain(facts, rules, query):
    facts = set(facts)
    step = 1
    while True:
        new_facts = set()
        for premises, conclusion in rules:
            # Find all matching substitutions for the premises
            substitutions = match_all_premises(premises, facts, {})
            for theta in substitutions:
                derived = substitute(conclusion, theta)
                if derived not in facts and derived not in new_facts:
                    print(f"[Step {step}] Derived: {derived} using {theta}")
                    new_facts.add(derived)
                    if derived == query:
                        print("\nQuery Reached successfully!")
                        return True
        if not new_facts:
            break
        facts.update(new_facts)
        step += 1
    return False

facts = [
    ("American", "West"),
    ("Enemy", "Nono", "America"),
    ("Owns", "Nono", "M1"),
    ("Missile", "M1")
]

rules = [
    # Missile(?x) => Weapon(?x)
    ([("Missile", "?x")], ("Weapon", "?x")),

    # Enemy(?x, America) => Hostile(?x)
    ([("Enemy", "?x", "America")], ("Hostile", "?x")),

    # Missile(?x) & Owns(Nono, ?x) => Sells(West, ?x, Nono)
    ([("Missile", "?x"), ("Owns", "Nono", "?x")], ("Sells", "West", "?x", "Nono")),

    # American(?x) & Weapon(?y) & Sells(?x, ?y, ?z) & Hostile(?z) => Criminal(?x)
    ([("American", "?x"), ("Weapon", "?y"), ("Sells", "?x", "?y", "?z"), ("Hostile", "?z")], ("Criminal", "?x"))
]

# Query
query = ("Criminal", "West")
forward_chain(facts, rules, query)

[Step 1] Derived: ('Weapon', 'M1') using {'?x': 'M1'}
[Step 1] Derived: ('Hostile', 'Nono') using {'?x': 'Nono'}
[Step 1] Derived: ('Sells', 'West', 'M1', 'Nono') using {'?x': 'M1'}
[Step 2] Derived: ('Criminal', 'West') using {'?x': 'West', '?y': 'M1', '?z': 'Nono'}

Query Reached successfully!


True